# Historical tweet behavior and later account status

This analysis uses **all 386,327 accounts with audited exact IDs and historical BotometerLite scores**. “Bot” means score **≥ 0.7** and “Human” means score **< 0.7** in the initial behavior and status comparisons; these are score-defined groups, not verified identities. There is no sampling or sampling weighting.

The historical tweet export covers **November 2021–December 2022** and concerns Ukraine/Russia. Scores come from the original companion BotometerLite CSV, linked through the audited exact-ID recovery. The only later variable is account status, taken from the saved **TwittrAPI observations on 11 November 2026 (Asia/Singapore)**. This notebook makes no API calls.

`account_data_clean_zachary.csv` is the sole input to the initial behavior and status comparisons. The appended tweet-type analysis also reads the historical tweet export and audited score file. Re-running this notebook displays the tables and charts inline without creating additional files.

## Field definitions and counting rules

| Field | Definition |
| --- | --- |
| `user_id` | Exact numeric-string historical author ID; joins use IDs, not current handles. |
| `botometerlite_score` | Unchanged historical score from the original companion score export. |
| `no_of_tweets` | Number of distinct tweets by this account observed in the historical topic dataset, including native retweets; not the profile's lifetime tweet count. |
| `no_of_tweets_with_urls` | Observed tweets with at least one URL entity or an HTTP(S) URL in their content text. Media links count as URLs when represented as such. |
| `no_of_tweets_with_media` | Observed tweets with media keys, media objects, or an exported photo/video/GIF type. A text URL alone does not establish attached media. |
| `no_of_tweets_with_mentions` | Observed tweets with a mention entity; if entities are missing, a textual @username match (1–15 ASCII letters, digits or underscores, ending at a word boundary) supplies evidence. A prefix of an overlong token does not count. |
| `no_of_followers`, `no_of_following` | Historical author profile counts attached to the account's latest observed tweet (timestamp, then numeric tweet ID as tie-breaker). This timestamp is not proof of profile-metadata collection time. |
| `uses_default_profile_pic` | Historical author image URL contains `/default_profile_images/`; `True`/`False`, or blank when no image URL is available. The export has no explicit default-picture boolean. |
| `current_account_status` | Saved ID-attributed status on 11 November 2026, preserving the source labels, including the single `lookup_error`. |

For **native retweets**, content is the embedded original post in `referenced_tweets`: its text, URL/mention entities and media attachments. This recovers information omitted from truncated outer retweet records and excludes the synthetic `RT @author` attribution from content mentions. Replies and quotes use their own exported fields and do not inherit the parent post's features. Each observed retweet still counts as one tweet by the retweeter. The builder stops on duplicate tweet IDs or failed retweet reconstruction rather than silently changing denominators.

A zero feature count means **no evidence in the available export**, not verified absence in a complete platform record. The 386,327-account cohort excludes unrecoverable or ambiguous original score records, as documented in `../../docs/provenance.md`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display

# Support running from either the repository root or this notebook directory.
HERE = Path.cwd()
if not (HERE / 'account_data_clean_zachary.csv').exists():
    HERE = HERE / 'results' / 'analysis'
CSV_PATH = HERE / 'account_data_clean_zachary.csv'
THRESHOLD = 0.7
GROUPS = ['Bot', 'Human']
COLORS = ['#D97732', '#367C9D']
EXPECTED_COLUMNS = [
    'user_id', 'botometerlite_score', 'no_of_tweets', 'no_of_tweets_with_urls',
    'no_of_tweets_with_media', 'no_of_tweets_with_mentions', 'no_of_followers',
    'no_of_following', 'uses_default_profile_pic', 'current_account_status',
]
data = pd.read_csv(CSV_PATH, dtype={
    'user_id': 'string', 'uses_default_profile_pic': 'boolean',
    'no_of_followers': 'Int64', 'no_of_following': 'Int64',
})
assert data.columns.tolist() == EXPECTED_COLUMNS
assert data.user_id.is_unique and data.user_id.str.fullmatch(r'\d+').all()
assert data.botometerlite_score.between(0, 1).all()
assert data.no_of_tweets.gt(0).all()
for column in EXPECTED_COLUMNS[3:6]:
    assert data[column].between(0, data.no_of_tweets).all()
assert data.current_account_status.notna().all()
data['group'] = np.where(data.botometerlite_score >= THRESHOLD, 'Bot', 'Human')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.titleweight': 'bold'})
print(f'{len(data):,} accounts; {data.no_of_tweets.sum():,} observed tweets')
display(data.head())
display(data[EXPECTED_COLUMNS].isna().sum().rename('Missing values').to_frame())

## Denominators

Tweet comparisons are **pooled proportions**: sum the feature-positive tweet counts within a group, then divide by the sum of its observed tweets. This is the requested tweet-weighted comparison, so accounts with more observed tweets contribute more; it is not the mean of per-account proportions.

Account-status proportions use **all accounts in each group** as the denominator. Default-picture proportions use accounts with an observed historical image URL; the table reports any missing values explicitly. When all image URLs are available, that denominator equals all accounts in the group.

In [ ]:
grouped = data.groupby('group', sort=False)
summary = grouped.agg(
    accounts=('user_id', 'size'),
    tweets=('no_of_tweets', 'sum'),
    tweets_with_urls=('no_of_tweets_with_urls', 'sum'),
    tweets_with_media=('no_of_tweets_with_media', 'sum'),
    tweets_with_mentions=('no_of_tweets_with_mentions', 'sum'),
    default_picture_accounts=('uses_default_profile_pic', 'sum'),
    known_picture_accounts=('uses_default_profile_pic', 'count'),
).reindex(GROUPS)
summary['missing_picture_accounts'] = summary.accounts - summary.known_picture_accounts
for feature in ['urls', 'media', 'mentions']:
    summary[f'{feature}_proportion'] = summary[f'tweets_with_{feature}'] / summary.tweets
summary['default_picture_proportion'] = summary.default_picture_accounts / summary.known_picture_accounts.replace(0, np.nan)
formatted_summary = summary.copy()
for column in summary.columns:
    pattern = '{:.2%}' if column.endswith('_proportion') else '{:,.0f}'
    formatted_summary[column] = summary[column].map(pattern.format)
display(formatted_summary)


def comparison_bar(ax, proportions, numerators, denominators, title, ylabel, ymax):
    values = np.asarray(proportions, dtype=float)
    bars = ax.bar(GROUPS, values, color=COLORS, width=0.58)
    ax.set(title=title, ylabel=ylabel, ylim=(0, ymax))
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.set_axisbelow(True)
    ax.grid(axis='y', alpha=0.18)
    for bar, value, numerator, denominator in zip(bars, values, numerators, denominators):
        if np.isfinite(value):
            ax.annotate(f'{value:.2%}\n{int(numerator):,} / {int(denominator):,}',
                        (bar.get_x() + bar.get_width()/2, value), xytext=(0, 6),
                        textcoords='offset points', ha='center', va='bottom', fontsize=10)
    ax.set_xlabel('Bot: score ≥ 0.7     |     Human: score < 0.7', fontsize=9)


def tweet_chart(feature, title, filename):
    fig, ax = plt.subplots(figsize=(7.2, 5.2), layout='constrained')
    comparison_bar(ax, summary[f'{feature}_proportion'], summary[f'tweets_with_{feature}'],
                   summary.tweets, title, 'Proportion of observed tweets', 1.0)
    plt.show()

## Proportion of Tweets Containing URLs

For each group: `sum(no_of_tweets_with_urls) / sum(no_of_tweets)`.

In [ ]:
tweet_chart('urls', 'Proportion of Tweets Containing URLs', 'tweets_with_urls.png')

## Proportion of Tweets Containing Media

For each group: `sum(no_of_tweets_with_media) / sum(no_of_tweets)`.

In [ ]:
tweet_chart('media', 'Proportion of Tweets Containing Media', 'tweets_with_media.png')

## Proportion of Tweets Containing Mentions

For each group: `sum(no_of_tweets_with_mentions) / sum(no_of_tweets)`.

In [ ]:
tweet_chart('mentions', 'Proportion of Tweets Containing Mentions', 'tweets_with_mentions.png')

## Proportion of Accounts Using Default Profile Picture

For each group: number of `True` values divided by number of observed historical picture indicators. Missing indicators remain unknown.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 5.2), layout='constrained')
comparison_bar(ax, summary.default_picture_proportion, summary.default_picture_accounts,
               summary.known_picture_accounts, 'Accounts Using Default Profile Picture',
               'Proportion of accounts with known picture indicator',
               max(0.05, float(summary.default_picture_proportion.max()) * 1.6))
plt.show()
print('Accounts with missing picture indicators:', int(summary.missing_picture_accounts.sum()))

## Accounts Active / Protected / Suspended / Deactivated / Unavailable (Reason Unknown)

Each panel shows the status count divided by **all accounts in its score group**, with the exact count and denominator printed above each bar. All panels use the same 0–100% scale for comparison.

**Active** means a public profile was returned, not that the account posted recently. **Protected** means an identified private profile. **Suspended** and **deactivated** require explicit provider messages. **Unavailable (reason unknown)** means a successful individual lookup returned no available profile without an attributable cause. These are later observations, not historical statuses during the tweet period.

The source also contains **one `lookup_error`** (a failed request after retries). It is preserved in the CSV and count table, included in its group's denominator, and is not relabeled as confirmed unavailability. Consequently, the five displayed status proportions can sum to slightly less than 100%.

In [ ]:
STATUSES = [
    ('active', 'Active'), ('protected', 'Protected'), ('suspended', 'Suspended'),
    ('deactivated', 'Deactivated'),
    ('unavailable_reason_unknown', 'Unavailable\n(reason unknown)'),
]
status_counts = pd.crosstab(data.group, data.current_account_status).reindex(
    index=GROUPS, columns=[s for s, _ in STATUSES] + ['lookup_error'], fill_value=0)
assert status_counts.sum(axis=1).equals(summary.accounts)
status_proportions = status_counts.div(summary.accounts, axis=0)
display(status_counts)
display(status_proportions.map('{:.2%}'.format))
fig, axes = plt.subplots(1, 5, figsize=(20, 5.2), sharey=True, layout='constrained')
for i, (ax, (status, title)) in enumerate(zip(axes, STATUSES)):
    comparison_bar(ax, status_proportions[status], status_counts[status], summary.accounts,
                   title, 'Proportion of all accounts' if i == 0 else '', 1.0)
    ax.set_xlabel('Bot ≥ 0.7 | Human < 0.7', fontsize=9)
fig.suptitle('Account Status on 11 November 2026 (Asia/Singapore)', fontsize=16)
plt.show()

## Reading these comparisons

These are descriptive comparisons of the full eligible historical cohort. The topic export is not a record of every tweet each account posted. BotometerLite groups are classifications based on scores; neither group supplies ground-truth automation labels. Historical follower/following and profile indicators may overlap the model's inputs. The temporal gap means later availability cannot establish automation or the reason for a historical behavior difference.

All tables and charts are embedded in this executed notebook. Re-running the notebook recomputes the preceding comparisons directly from `account_data_clean_zachary.csv`; the appended section documents its additional inputs.

Validation reconciles all 8,000 earlier sampled accounts. Tweet totals, URL/media counts, follower/following counts, and default-picture indicators agree exactly. Mention counts differ for two accounts (three tweets): the earlier fallback matched prefixes of the overlong tokens `@ThreatOfNuclearWar` and `@UkrainianInvasion`. This analysis requires a complete username token of at most 15 characters, so those unsupported mentions are excluded.

## Unique accounts by tweet type and bots among original-posting accounts

This section uses the full historical tweet export, **not** the earlier 386,327-account scored cohort. The requested `English_100.csv` is not present locally; the source used here is `../../data/raw/English_100_vectionary_scored.csv`, the file discussed above. Accounts are distinct exact-string **`author_id`** values, checked against `author.id`; handles are not used as identities.

The CSV is read with `lineterminator="\n"`. Embedded carriage returns otherwise cause Python's default CSV reader to split some records incorrectly. This corrects the earlier conversational counts: the export has **945,686 tweets**, not 945,881 nonblank records, and the correctly parsed `referenced_tweets.type` column has **no missing values**. The code nevertheless omits missing or whitespace-only tweet types as requested and reports the exclusion count. The all-account denominator is computed before exclusions.

Each distinct exported type is reported separately, including the combined value `quoted replied_to`. An account is counted once per type it posted, so accounts may appear in several rows and percentages need not sum to 100%. “Original” means the exact value `original`.

Historical scores are joined by exact author ID to the audited `../../data/bot_scores_recovered.csv`. Here, **bot means score ≥ 0.7**, matching the threshold in the earlier comparisons. The bot percentage denominator is **all distinct original-posting accounts**, including accounts without an audited score. Score coverage is reported separately; missing scores are unknown, not classified as nonbots. This is a score-based classification, not verified automation status.


In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
from IPython.display import display

# This section can run independently from the preceding notebook cells.
analysis_root = next(
    candidate for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / 'data/raw/English_100_vectionary_scored.csv').is_file()
)
tweet_source = analysis_root / 'data/raw/English_100_vectionary_scored.csv'
score_source = analysis_root / 'data/bot_scores_recovered.csv'
all_tweet_accounts = set()
accounts_by_tweet_type = defaultdict(set)
tweets_by_type = Counter()
tweet_rows = missing_type_rows = 0
expected_types = {'original', 'replied_to', 'quoted', 'retweeted', 'quoted replied_to'}

for chunk in pd.read_csv(
    tweet_source,
    usecols=['author_id', 'author.id', 'referenced_tweets.type'],
    dtype='string', lineterminator='\n', chunksize=100_000,
):
    assert chunk['author_id'].notna().all()
    assert chunk['author_id'].str.fullmatch(r'\d+').all()
    assert chunk['author_id'].equals(chunk['author.id'])
    tweet_rows += len(chunk)
    all_tweet_accounts.update(chunk['author_id'])
    types = chunk['referenced_tweets.type'].str.strip()
    keep = types.notna() & types.ne('').fillna(False)
    missing_type_rows += int((~keep).sum())
    valid = chunk.loc[keep, ['author_id']].assign(tweet_type=types[keep])
    assert set(valid['tweet_type']).issubset(expected_types)
    for tweet_type, rows in valid.groupby('tweet_type', sort=False):
        accounts_by_tweet_type[tweet_type].update(rows['author_id'])
        tweets_by_type[tweet_type] += len(rows)

assert tweet_rows == sum(tweets_by_type.values()) + missing_type_rows
assert set().union(*accounts_by_tweet_type.values()).issubset(all_tweet_accounts)
total_tweet_accounts = len(all_tweet_accounts)
assert total_tweet_accounts > 0

print(f'Source: {tweet_source.relative_to(analysis_root)}')
print(f'Total tweets: {tweet_rows:,}')
print(f'Tweets omitted for missing/blank type: {missing_type_rows:,}')
print(f'Total distinct Twitter accounts: {total_tweet_accounts:,}')

type_order = ['original', 'replied_to', 'quoted', 'retweeted', 'quoted replied_to']
tweet_type_account_summary = pd.DataFrame([
    {'Tweet type': kind,
     'Number': len(accounts_by_tweet_type[kind]),
     'Percentage of all unique accounts': 100 * len(accounts_by_tweet_type[kind]) / total_tweet_accounts}
    for kind in type_order if kind in accounts_by_tweet_type
]).set_index('Tweet type')
display(tweet_type_account_summary.assign(**{
    'Number': tweet_type_account_summary['Number'].map('{:,.0f}'.format),
    'Percentage of all unique accounts': tweet_type_account_summary['Percentage of all unique accounts'].map('{:.2f}%'.format),
}))

historical_scores = pd.read_csv(score_source, dtype={'user_id': 'string'})
assert historical_scores['user_id'].notna().all()
assert historical_scores['user_id'].is_unique
assert historical_scores['botometerlite_score'].between(0, 1).all()
assert set(historical_scores['user_id']).issubset(all_tweet_accounts)
original_posting_accounts = pd.DataFrame({
    'user_id': sorted(accounts_by_tweet_type['original'])
}).merge(
    historical_scores[['user_id', 'botometerlite_score']],
    on='user_id', how='left', validate='one_to_one',
)
original_accounts_total = len(original_posting_accounts)
assert original_accounts_total > 0
original_scores = original_posting_accounts['botometerlite_score']
original_bot_count = int(original_scores.ge(0.7).sum())
original_score_coverage = pd.DataFrame({
    'Number': [original_accounts_total, int(original_scores.notna().sum()),
               int(original_scores.isna().sum()), int(original_scores.eq(0.7).sum()),
               original_bot_count],
}, index=[
    'All original-posting accounts', 'With an audited bot score',
    'Without an audited bot score (unknown)', 'Score exactly 0.7 (included in bots)',
    'Bots: score ≥ 0.7',
])
original_score_coverage['Percentage of all original-posting accounts'] = (
    100 * original_score_coverage['Number'] / original_accounts_total
)
assert (original_scores.lt(0.7).sum() + original_bot_count + original_scores.isna().sum()) == original_accounts_total
display(original_score_coverage.assign(**{
    'Number': original_score_coverage['Number'].map('{:,.0f}'.format),
    'Percentage of all original-posting accounts': original_score_coverage['Percentage of all original-posting accounts'].map('{:.2f}%'.format),
}))
print(f'Bots among original-posting accounts: {original_bot_count:,} / '
      f'{original_accounts_total:,} = {original_bot_count / original_accounts_total:.4%}')


Source: data/raw/English_100_vectionary_scored.csv
Total tweets: 945,686
Tweets omitted for missing/blank type: 0
Total distinct Twitter accounts: 390,804


,Number,Percentage of all unique accounts
Tweet type,,
original,"55,429",14.18%
replied_to,"46,725",11.96%
quoted,"6,666",1.71%
retweeted,"313,659",80.26%
quoted replied_to,418,0.11%


,Number,Percentage of all original-posting accounts
All original-posting accounts,"55,429",100.00%
With an audited bot score,"54,764",98.80%
Without an audited bot score (unknown),665,1.20%
Score exactly 0.7 (included in bots),134,0.24%
Bots: score ≥ 0.7,"2,796",5.04%


Bots among original-posting accounts: 2,796 / 55,429 = 5.0443%
